In [ ]:
%python
# 1. Check Spark Session & Version
print("Spark version:", spark.version)

# 2. Create sample DataFrame
data = [
    ("Delhi", 420),
    ("Gurugram", 350),
    ("Noida", 380),
    ("Mumbai", 110),
    ("Pune", 85),
    ("Jaipur", 240)
]

df = spark.createDataFrame(data, ["city", "aqi"])

# 3. Display table in Databricks
display(df)

In [ ]:
#Get Started
# Driver Node (Master Node & commander)
# Builder(Build plan) -- part of driver node 
# Worker Node (Executors who run the plan parallely across cluster) and each machine who part of cluster is worker node that coordinate with master node for execution
#Executors part of worker node who execute thing 

# nodes classify as driver and worker and the task they perform are called as executors and builders and the plan is called as DAG

# Master slave architecture
# cluster mamager is the one who manage the cluster and the worker node and the driver node


In [ ]:
spark.version

In [ ]:
df = spark.createDataFrame([(1,"sidhart", 25), (2, "Amit", 30), (3, "Ravi", 40), (4, "Neha", 28)], ["id", "name", "age"]) # DataFrame actually not created Lazy it just added in plan and inserted in DAG (directed acylic graph)
df.show() # Actions that actually make enforce the worker to execute the plan by builder who made in the driver node

In [ ]:
spark.sql("CREATE VOLUME IF NOT EXISTS aeris_brick.default.apache_spark_data")
output_path = "/Volumes/aeris_brick/default/apache_spark_data/ApacheSpark"
df.write.mode("overwrite").csv(output_path)
df.write.mode("overwrite").option("header", "true").csv(output_path + "/sample.csv")
# Modern azure databricks uses volume/catalog/schema/volume_name
# schema == database
# catalog == org 


In [ ]:
df.write.mode("overwrite").option("header", "true").saveAsTable("aeris_brick.default.sample_table")

In [ ]:
display(df)

In [ ]:
df = spark.read.csv("/Volumes/aeris_brick/default/apache_spark_data/datasets/Employee.csv", header = True, inferSchema=True) #inferSchema preserve the data type of the data based on the data
display(df)

In [ ]:
# *unpack operator which unpack everything and pass whateer in list as separate stringh or whatever it as argument

In [ ]:
df.show(3)
display(df.limit(3)) # other way to show using display
df.limit(3)

In [ ]:
display(df.describe()) # or df.describe().show() 
#both show and display did same job

In [ ]:
df.printSchema() #inferschema real in action

In [ ]:
df.columns

In [ ]:
print(*df.columns)

In [ ]:
df.count()

In [ ]:
from pyspark.sql.functions import col
df.select("Age").show()
df.select("Age","Education").show()
df.select(col("Age").alias("DOB")).show()

In [ ]:
df.selectExpr("*","Age+1 as AgePlusOne").show()

In [ ]:
df.selectExpr("Age+1 as AgePlusOne").show()

In [ ]:
df = spark.read.table("samples.bakehouse.sales_customers")
display(df)
display(df.printSchema())

In [ ]:
from pyspark.sql.functions import udf
@udf("double")
def sum(a, b):
    return int(a) + int(b)
df.withColumn("sum", sum("customerID", "phone_number")).show()

In [ ]:
spark.read.table("samples.bakehouse.sales_transactions").show()

In [ ]:
# We need to create a new discount column on our profit but based on the differnce categories in payment mode column 
# Upi net banking = 7 %
# cash delivery = 5 %
# credit card = 10 %


In [ ]:
%sql
-- Create a table directly by reading the CSV file in your volume
CREATE TABLE IF NOT EXISTS default.sales_table
AS SELECT * FROM read_files(
  '/Volumes/aeris_brick/default/apache_spark_data/ApacheSpark/sales.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

In [ ]:
df = spark.read.table("aeris_brick.default.sales_table")
display(df)

In [ ]:
from pyspark.sql.functions import udf, udtf
from pyspark.sql.types import *
# @udf("schema")
# def get_schema():
#   pass
# @udtf("double")
# def func():
#   pass
# df.display()

### The udf create a separate python runtime environment (Interpreter) on every worker node to just execute the following instruction which is why udf generally not recommended and if we not use udf then things run as per standard python runtime execution and logical expression where true or false we want python want immediate resolution of that stuff what we write and as using any thing using spark cause issue as it all happen in lazy execution so refernce is pass which python can't validate because at under hood everything run on jvm and python is just a api so never ever use native python instruction

In [ ]:
df.select("sales","category").display()

In [ ]:
from pyspark.sql.functions import col
df.select((col("sales")+1).alias("sales_1")).display()# no need to confuse col or other method is just made to return object and it uses operator overloading underhood and select can accept string or object just the object we pass the column name we specify must belong to df and other is as it is simple and with inheritance or instance extends it become everythign connected
# selecExpr is allow to use sql directly like select age + 1 as t from studnet the diffence is select we alreay tell we use and from student df already tell so only pass age + 1 as t is more than enough

df.selectExpr("sales+1 as ageOne","category").display()

In [ ]:
df.withColumn("unit_price", (col("sales")/col("quantity")).cast("int")).display()

In [ ]:
df.filter(col("profit")>10000).display()
df.select(col("profit")>10).display()
df.selectExpr("profit>10").display()

In [ ]:
df.filter("profit > (SELECT AVG(profit) FROM aeris_brick.default.sales_table)").display()

In [ ]:
# df.where() it is equivalent to df.filter()
df.where(col("profit")>10000).display() 
df.where("profit>10000").display()
# both are literally same 

In [ ]:
df.filter("profit>100 and quantity>1").display()
df.filter((col("profit")>100) & (col("quantity")>1)).display()

In [ ]:
df.filter((col("_rescued_data").isNotNull())).display()
df.filter((col("_rescued_data").isNull())).display()

In [ ]:
df.filter(col("customer_id").like("CUST_1%")).display()

In [ ]:
# When and otherwise
from pyspark.sql.functions import when
df.select(col("sales"),when(col("sales")>10000,"High").when(col("sales")>5000,"Medium").otherwise("Low").alias("sp")).display()
# withColumnRenamed
df.withColumnRenamed("sales","sales_1").display()

df.withColumn("sp",when(col("sales")>10000,"High").when(col("sales")>5000,"Medium").otherwise("Low").alias("sp")).display()

In [ ]:
df.selectExpr("sales", "CASE WHEN sales > 10000 THEN 'High' WHEN sales > 5000 THEN 'Medium' ELSE 'Low' END AS sp").display()

In [ ]:
df.printSchema()

In [ ]:
from pyspark.sql.types import *
schema = StructType([
    StructField("order_id", StringType(),False), # True defines nullability if it is true we are allow to hold null values
    StructField("order_date", IntegerType()),
    StructField("customer_id", StringType()),
    StructField("product_id", StringType())
])

data = [
    (1, None, None, None),
    (1, 2, 3, 4),
    (5, 6, 7, 8)
]

df = spark.createDataFrame(data, schema = schema)
df.display()

In [ ]:
df = spark.read.table("aeris_brick.default.sales_table")
display(df)

In [ ]:
df.display(3)

In [ ]:
df.show(20, truncate=True, vertical=True)

In [ ]:
df.limit(5).display()

In [ ]:
# UDF AND UDTF
from pyspark.sql.functions import count
@udf(returnType=StringType())
def salary_label(salary):
    if salary is None :
        return "Unknown"
    return "High" if salary>=50000 else "Normal"
df.withColumn("salary_label", salary_label(col("sales"))).display()
df.filter(col("sales").isNull()).count()
df.selectExpr("count(case when sales is null then 1 end) as s").display()

In [ ]:
from pyspark.sql.functions import udtf, lit, col, avg
@udtf(returnType="number: int, square: int")
class GenerateSquares:
    def eval(self, n):
        print("eval received:", n)
        for i in range(1, n + 1):
            yield (i, i * i)

GenerateSquares(lit(3)).show()
a = 3
b = lit(3)
print(a,b)
print(type(a), type(b))
print(col("sales"))
df.printSchema()
df.selectExpr("avg(quantity)").display()
df.select(avg(col("quantity"))).display()

In [ ]:
from pyspark.sql.functions import count, sum, avg
df.groupBy("city").count().show()
df.groupBy("city").agg(
    count("*").alias("total_orders"),
    sum("sales").alias("total_sales"),
    avg("profit").alias("avg_profit")
).display()

In [ ]:
employees.join(
    departments,
    on = (employees["dept_id"] == departments["cust_id"]) & (employees["city"] == departments["city"]),
    how = "inner"
).show()

#or

employees = spark.createDataFrame(
    [(1, "Amit", 10), (2, "Neha", 20)],
    ["id", "name", "dept_id"]
)

departments = spark.createDataFrame(
    [(10, "IT"), (20, "HR")],
    ["dept_id", "department"]
)

employees.join(
    departments,
    on="dept_id",
    how="inner"
).show()

In [ ]:

employees.createOrReplaceTempView("emp") #employee is dataframe
departments.createOrReplaceTempView("dept") #department is dataframe

spark.sql("""
    SELECT e.name, d.department
    FROM emp e
    INNER JOIN dept d
        ON e.dept_id = d.dept_id
""").show()
